# Experiment 10 — Buyer-scheme distinctiveness (v15-specific)

**Paper reference.** §2.2 of v15 (the four buyer schemes A–D are probes, not behaviours under study).

**Goal.** The seller's policy should depend only on the *content* of the buyer's offer, not which sampling rule produced it. Train a linear model under one buyer scheme and test under another, holding the seller policy fixed. Off-diagonal entries close to the diagonal would confirm that the seller policy is scheme-invariant.

In [ ]:
import os, sys, warnings
warnings.simplefilter('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
import negolin_v15 as nv
nv.setup_matplotlib()

DATA_DIR = '/home/claude/data/merged_results'    # <-- edit if needed
OUT_DIR  = './out'
os.makedirs(OUT_DIR, exist_ok=True)

raw = nv.load_run_files(DATA_DIR)
_, turns = nv.preprocess_turns(raw)
POLICIES = list(nv.POLICIES)
SCHEMES  = list(nv.SCHEMES)
print(f'rows={len(turns):,}  policies={POLICIES}  schemes={SCHEMES}')

In [ ]:
# Load the Stage-2 selection produced by 02_main_sweep.ipynb
sel_path = Path(OUT_DIR) / 'selected_linear.csv'
if sel_path.exists():
    SELECTED = pd.read_csv(sel_path)
    print('Loaded selected_linear.csv:'); display(SELECTED.head())
else:
    # Sensible fallback: Feature II + Ridge per (scheme, policy)
    rows = [{'scheme': s, 'policy': p, 'chosen_family': 'II', 'chosen_model': 'Ridge'}
            for s in SCHEMES for p in POLICIES]
    SELECTED = pd.DataFrame(rows)
    print('selected_linear.csv not found -> using fallback (II/Ridge)')

def picked(scheme, policy):
    r = SELECTED.set_index(['scheme','policy']).loc[(scheme, policy)]
    fam = r['chosen_family']
    if fam == 'Base': fam = 'II'    # Base isn't a feature spec -- fall back
    return fam, r['chosen_model']

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

COMMON_FAM = 'II'

def design_for_scheme(scheme, policy, fam=COMMON_FAM):
    sub = turns.query('scheme == @scheme and policy == @policy')
    meta, X, y, _ = nv.build_design_matrix(sub, fam)
    (_, _, _), (m_n, X_n, y_n) = nv.partition_persist(meta, X, y)
    return m_n, X_n, y_n

def transfer_matrix_schemes(policy):
    designs = {s: design_for_scheme(s, policy) for s in SCHEMES}
    mat = pd.DataFrame(np.nan, index=SCHEMES, columns=SCHEMES, dtype=float)
    for s_tr in SCHEMES:
        m_tr, X_tr, y_tr = designs[s_tr]
        if len(y_tr) < 30: continue
        est = Pipeline([('s', StandardScaler()), ('m', Ridge(alpha=1.0))]).fit(X_tr, y_tr)
        for s_te in SCHEMES:
            m_te, X_te, y_te = designs[s_te]
            if len(y_te) < 5: continue
            mat.loc[s_tr, s_te] = float(np.mean(np.abs(est.predict(X_te) - y_te)))
    return mat

fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for ax, pol in zip(axes.ravel(), POLICIES):
    M = transfer_matrix_schemes(pol)
    im = ax.imshow(M.values, cmap='viridis_r', aspect='auto')
    ax.set_title(f'{pol}')
    ax.set_xticks(range(len(SCHEMES))); ax.set_xticklabels(SCHEMES)
    ax.set_yticks(range(len(SCHEMES))); ax.set_yticklabels(SCHEMES)
    for i in range(len(SCHEMES)):
        for j in range(len(SCHEMES)):
            v = M.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7,
                        color='white' if v > np.nanmean(M.values) else 'black')
    fig.colorbar(im, ax=ax, fraction=0.04)
fig.suptitle(f'Train on row scheme, test on column scheme  (Y-scale MAE, family {COMMON_FAM})', fontsize=11)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'fig_exp10_scheme_transfer.png'))
plt.show()

**Reading.** If diagonals dominate, schemes have noticeable signatures and the seller is reacting to *how* the buyer behaves, not just *what* she offered. If the matrix is roughly uniform with low off-diagonals, the seller is genuinely scheme-invariant — exactly what you want a 'seller policy' to mean.

## Per-cell: how much does scheme transfer cost on the dollar scale?

In [ ]:
# Off-diag mean - diag mean per policy: how much MAE we lose by training on the wrong scheme
rows = []
for pol in POLICIES:
    M = transfer_matrix_schemes(pol)
    diag = np.nanmean(np.diag(M.values))
    off  = np.nanmean(M.values - np.diag(np.diag(M.values)))
    rows.append({'policy': pol, 'diag_mean': diag, 'off_mean': off,
                 'gap': off - diag, 'rel_gap': (off - diag) / diag if diag > 0 else np.nan})
pd.DataFrame(rows).round(4)